In [ ]:
import sys; from pathlib import Path
src_dir = next(parent / 'src' for parent in Path().absolute().parents if (parent / 'src').is_dir())
sys.path.extend([str(src_dir), str(src_dir / 'pipelines')])
from pipeline_particle_tracking import *; from imports import * ; current_dir = Path().resolve()

In [ ]:
# Modeling
model_path = Path('/Users/nzlab-la/Desktop/micro/modeling/TASEP')
sys.path.append(str(model_path))

from TASEP_models import *
#tag_dict = {'GFP': GFP_TAG, 'HA': HA_TAG, 'U': U_TAG, 'SUN': SUN_TAG, 'ALFA': ALFA_TAG}
tag_sequence = tag_dict['SUN']

#dna_file_path = pathlib.Path( '/Users/nzlab-la/Desktop/micro/gene_sequences/utag_project/pNZ208(pUB-24xUTagFullLength-KDM5B-MS2).dna' )   # UTAG
dna_file_path = pathlib.Path( '/Users/nzlab-la/Desktop/micro/gene_sequences/utag_project/pNZ266(pUB-24xGCN4-KDM5B-MS2).dna' )             # SunTag
#dna_file_path = pathlib.Path( '/Users/nzlab-la/Desktop/micro/gene_sequences/utag_project/pNZ267 (pUB-24xALFAtag-KDM5B-MS2).dna' )         # AlfaTag



In [ ]:
# reading the sequence and extracting the elongation rates
protein, rna, dna, indexes_tags, _, seq_record, graphic_features  = read_sequence(seq=dna_file_path, min_protein_length=50,TAG=[tag_sequence])
plasmid_figure = plot_plasmid(seq_record, graphic_features,figure_width=25, figure_height=3)

gene_length = len(protein)+1 # adding 1 to account for the stop codon
tag_positions_first_probe_vector = indexes_tags[0]
tag_positions_second_probe_vector = indexes_tags[1] if len(indexes_tags) > 1 else None

first_probe_position_vector = create_probe_vector(tag_positions_first_probe_vector, gene_length)
second_probe_position_vector = create_probe_vector(tag_positions_second_probe_vector, gene_length) if tag_positions_second_probe_vector is not None else None


In [ ]:
# # Initial conditions
ki = 0.03  # Initiation rate
elongation_rate_constant = 5  # Elongation rates for positions 1 to N-1
number_repetitions = 50
burnin_time = 1000
t_max = 360*5 #timePerturbationApplication + 25*60  # Maximum time
step_size_in_sec = 5 # 5
#ke = calculate_codon_elongation_rates (rna, global_elongation_rate=global_elongation_rate)


In [ ]:
ke = calculate_codon_elongation_rates (rna, global_elongation_rate=elongation_rate_constant)


In [ ]:
intensity_vector_first_signal_ode,intensity_vector_second_signal_ode = simulate_TASEP_ODE(ki, ke, gene_length, t_max,first_probe_position_vector,second_probe_position_vector,burnin_time, time_interval_in_seconds= step_size_in_sec)


In [ ]:
time_array = np.arange(0, t_max, step_size_in_sec)

In [ ]:
constant_elongation_rate = None
ssa_complete = simulate_TASEP_SSA(ki, ke, gene_length, t_max,
                                time_interval_in_seconds=step_size_in_sec,
                                number_repetitions=number_repetitions, 
                                first_probe_position_vector=first_probe_position_vector, 
                                second_probe_position_vector=second_probe_position_vector,
                                burnin_time=burnin_time,
                                constant_elongation_rate=constant_elongation_rate,
                                 ) [2]
# 14.6 sec

In [ ]:
constant_elongation_rate = elongation_rate_constant
ssa_constant = simulate_TASEP_SSA(ki, ke, gene_length, t_max,
                            time_interval_in_seconds=step_size_in_sec,
                            number_repetitions=number_repetitions, 
                            first_probe_position_vector=first_probe_position_vector, 
                            second_probe_position_vector=second_probe_position_vector,
                            burnin_time=burnin_time,
                            constant_elongation_rate=elongation_rate_constant,
                            ) [2]
# 12.4seconds

In [ ]:
constant_elongation_rate = elongation_rate_constant
ssa_constant_fo = simulate_TASEP_SSA(ki, ke, gene_length, t_max,
                            time_interval_in_seconds=step_size_in_sec,
                            number_repetitions=number_repetitions, 
                            first_probe_position_vector=first_probe_position_vector, 
                            second_probe_position_vector=second_probe_position_vector,
                            burnin_time=burnin_time,
                            constant_elongation_rate=elongation_rate_constant,
                            fast_output=False,
                            ) [2]
# 12.7 sec


In [ ]:
# model waiting time for two initiation step model
ki_pre = 0.06
ki_ini = 0.06
effective_initiation_rate = (ki_pre * ki_ini) / (ki_pre + ki_ini)
ki_waiting_time = 1 / effective_initiation_rate
print('ki eff: ', effective_initiation_rate)
print('waiting time: ', ki_waiting_time )

In [ ]:
# simulating multi-step initiation
constant_elongation_rate = elongation_rate_constant

ki_multi_step = [ki_pre, ki_ini]

ssa_constant_2_step_ki = simulate_TASEP_SSA(ki_multi_step, ke, gene_length, t_max,
                            time_interval_in_seconds=step_size_in_sec,
                            number_repetitions=number_repetitions, 
                            first_probe_position_vector=first_probe_position_vector, 
                            second_probe_position_vector=second_probe_position_vector,
                            burnin_time=burnin_time,
                            constant_elongation_rate=6,
                            fast_output=True,
                            #multi_step_initiation_model = True,
                            ) [2]

In [ ]:
# Three-step initiation parameters
ki1 = 0.02
ki2 = 0.02
ki3 = 0.02
# Calculate effective waiting time and effective initiation rate
T_wait = 1/ki1 + 1/ki2 + 1/ki3
k_eff = 1 / T_wait
print("Effective initiation rate:", k_eff)
print("Waiting time:", T_wait)

In [ ]:
# simulating multi-step initiation
constant_elongation_rate = elongation_rate_constant
ki_multi_step = [ki1, ki2, ki3]
ssa_constant_3_step_ki = simulate_TASEP_SSA(ki_multi_step, ke, gene_length, t_max,
                            time_interval_in_seconds=step_size_in_sec,
                            number_repetitions=number_repetitions, 
                            first_probe_position_vector=first_probe_position_vector, 
                            second_probe_position_vector=second_probe_position_vector,
                            burnin_time=burnin_time,
                            constant_elongation_rate=6,
                            fast_output=True,
                            #multi_step_initiation_model = True,
                            ) [2]

In [ ]:
# Calculate SEM
sem_ssa_complete = np.std(ssa_complete, axis=0) / np.sqrt(number_repetitions)
sem_ssa_constant = np.std(ssa_constant, axis=0) / np.sqrt(number_repetitions)
sem_ssa_constant_fo = np.std(ssa_constant_fo, axis=0) / np.sqrt(number_repetitions)
sem_ssa_constant_2_step_ki = np.std(ssa_constant_2_step_ki, axis=0) / np.sqrt(number_repetitions)
sem_ssa_constant_3_step_ki = np.std(ssa_constant_3_step_ki, axis=0) / np.sqrt(number_repetitions)

plt.figure(figsize=(10, 5))
plt.plot(time_array, intensity_vector_first_signal_ode.astype(int), 'k', label='ODE First Signal')
plt.plot(time_array, np.mean(ssa_complete, axis=0), 'm', label='SSA Complete')
plt.fill_between(time_array, np.mean(ssa_complete, axis=0) - sem_ssa_complete, np.mean(ssa_complete, axis=0) + sem_ssa_complete, color='m', alpha=0.3, label='SEM SSA Complete')
plt.plot(time_array, np.mean(ssa_constant, axis=0), 'r', label='SSA 1')
plt.fill_between(time_array, np.mean(ssa_constant, axis=0) - sem_ssa_constant, np.mean(ssa_constant, axis=0) + sem_ssa_constant, color='r', alpha=0.3, label='SEM SSA 1')

plt.plot(time_array, np.mean(ssa_constant_fo, axis=0), 'b', label='SSA fo')
plt.fill_between(time_array, np.mean(ssa_constant_fo, axis=0) - sem_ssa_constant_fo, np.mean(ssa_constant_fo, axis=0) + sem_ssa_constant_fo, color='b', alpha=0.3, label='SEM SSA fo')

plt.plot(time_array, np.mean(ssa_constant_2_step_ki, axis=0), 'g', label='SSA 2-step ki')
plt.fill_between(time_array, np.mean(ssa_constant_2_step_ki, axis=0) - sem_ssa_constant_2_step_ki, np.mean(ssa_constant_2_step_ki, axis=0) + sem_ssa_constant_2_step_ki, color='g', alpha=0.3, label='SEM SSA 2-step ki')


plt.plot(time_array, np.mean(ssa_constant_3_step_ki, axis=0), 'c', label='SSA 3-step ki')
plt.fill_between(time_array, np.mean(ssa_constant_3_step_ki, axis=0) - sem_ssa_constant_3_step_ki, np.mean(ssa_constant_3_step_ki, axis=0) + sem_ssa_constant_3_step_ki, color='c', alpha=0.3, label='SEM SSA 3-step ki')


plt.legend()
plt.show()

In [ ]:
# compare ACF

mean_correlation_ssa, std_correlation_ssa, lags_ssa, correlations_array_ssa, dwell_time_ssa = mi.Correlation(primary_data=ssa_complete,
                                                                                        max_lag=None, 
                                                                                        nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                        shift_data=True,
                                                                                        return_full=False,
                                                                                        time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                        use_bootstrap=True,
                                                                                        show_plot=False,
                                                                                        start_lag=0,
                                                                                        fit_type='linear',
                                                                                        index_max_lag_for_fit = 100,
                                                                                        de_correlation_threshold=0.01,
                                                                                        correct_baseline=True,
                                                                                        use_linear_projection_for_lag_0=True,
                                                                                        save_plots=False,
                                                                                        remove_outliers = False,
                                                                                        outlier_percentile = 100,
                                                                                        plot_title=None,).run()


In [ ]:
mean_correlation_ssa_constant, std_correlation_ssa_constant, lags_ssa_constant, correlations_array_ssa_constant, dwell_time_ssa_constant = mi.Correlation(primary_data=ssa_constant,
                                                                                        max_lag=None, 
                                                                                        nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                        shift_data=True,
                                                                                        return_full=False,
                                                                                        time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                        use_bootstrap=True,
                                                                                        show_plot=False,
                                                                                        start_lag=0,
                                                                                        fit_type='linear',
                                                                                        index_max_lag_for_fit = 100,
                                                                                        de_correlation_threshold=0.01,
                                                                                        correct_baseline=True,
                                                                                        use_linear_projection_for_lag_0=True,
                                                                                        save_plots=False,
                                                                                        remove_outliers = False,
                                                                                        outlier_percentile = 100,
                                                                                        plot_title=None).run()


In [ ]:
mean_correlation_ssa_constant_fo, std_correlation_ssa_constant_fo, lags_ssa_constant, correlations_array_ssa_constant, dwell_time_ssa_constant = mi.Correlation(primary_data=ssa_constant_fo,
                                                                                        max_lag=None, 
                                                                                        nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                        shift_data=True,
                                                                                        return_full=False,
                                                                                        time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                        use_bootstrap=True,
                                                                                        show_plot=False,
                                                                                        start_lag=0,
                                                                                        fit_type='linear',
                                                                                        index_max_lag_for_fit = 100,
                                                                                        de_correlation_threshold=0.01,
                                                                                        correct_baseline=True,
                                                                                        use_linear_projection_for_lag_0=True,
                                                                                        save_plots=False,
                                                                                        remove_outliers = False,
                                                                                        outlier_percentile = 100,
                                                                                        plot_title=None).run()

In [ ]:
# calculate the correlation for the 2-step initiation model
mean_correlation_ssa_constant_2_step_ki, std_correlation_ssa_constant_2_step_ki, lags_ssa_constant, correlations_array_ssa_constant, dwell_time_ssa_constant = mi.Correlation(primary_data=ssa_constant_2_step_ki,
                                                                                        max_lag=None, 
                                                                                        nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                        shift_data=True,
                                                                                        return_full=False,
                                                                                        time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                        use_bootstrap=True,
                                                                                        show_plot=False,
                                                                                        start_lag=0,
                                                                                        fit_type='linear',
                                                                                        index_max_lag_for_fit = 100,
                                                                                        de_correlation_threshold=0.01,
                                                                                        correct_baseline=True,
                                                                                        use_linear_projection_for_lag_0=True,
                                                                                        save_plots=False,
                                                                                        remove_outliers = False,
                                                                                        outlier_percentile = 100,
                                                                                        plot_title=None).run()


In [ ]:
# calculate the correlation for the 3-step initiation model
mean_correlation_ssa_constant_3_step_ki, std_correlation_ssa_constant_3_step_ki, lags_ssa_constant, correlations_array_ssa_constant, dwell_time_ssa_constant = mi.Correlation(primary_data=ssa_constant_3_step_ki,
                                                                                        max_lag=None, 
                                                                                        nan_handling='forward_fill',  #forward_fill, 'ignore'
                                                                                        shift_data=True,
                                                                                        return_full=False,
                                                                                        time_interval_between_frames_in_seconds=step_size_in_sec,
                                                                                        use_bootstrap=True,
                                                                                        show_plot=False,
                                                                                        start_lag=0,
                                                                                        fit_type='linear',
                                                                                        index_max_lag_for_fit = 100,
                                                                                        de_correlation_threshold=0.01,
                                                                                        correct_baseline=True,
                                                                                        use_linear_projection_for_lag_0=True,
                                                                                        save_plots=False,
                                                                                        remove_outliers = False,
                                                                                        outlier_percentile = 100,
                                                                                        plot_title=None).run()

In [ ]:
start_lag =0
maxlag = 300

plt.figure(figsize=(10, 5))
plt.plot(lags_ssa[start_lag:maxlag], mean_correlation_ssa[start_lag:maxlag], 'o-', color='blue', linewidth=2, label='SSA', alpha=0.5)
plt.fill_between(lags_ssa[start_lag:maxlag], 
                mean_correlation_ssa[start_lag:maxlag] - std_correlation_ssa[start_lag:maxlag], 
                mean_correlation_ssa[start_lag:maxlag] + std_correlation_ssa[start_lag:maxlag], 
                color='blue', alpha=0.9)
# plot the correlation for the SSA
plt.plot(lags_ssa[start_lag:maxlag], mean_correlation_ssa_constant[start_lag:maxlag], 'o-', color='red', linewidth=2, label='SSA constant', alpha=0.5)
plt.fill_between(lags_ssa[start_lag:maxlag], 
                mean_correlation_ssa_constant[start_lag:maxlag] - std_correlation_ssa_constant[start_lag:maxlag], 
                mean_correlation_ssa_constant[start_lag:maxlag] + std_correlation_ssa_constant[start_lag:maxlag], 
                color='red', alpha=0.9)

plt.plot(lags_ssa[start_lag:maxlag], mean_correlation_ssa_constant_2_step_ki[start_lag:maxlag], 'o-', color='k', linewidth=2, label='SSA 2_step_ki', alpha=0.5)

plt.fill_between(lags_ssa[start_lag:maxlag],
                mean_correlation_ssa_constant_2_step_ki[start_lag:maxlag] - std_correlation_ssa_constant_2_step_ki[start_lag:maxlag],
                mean_correlation_ssa_constant_2_step_ki[start_lag:maxlag] + std_correlation_ssa_constant_2_step_ki[start_lag:maxlag],
                color='k', alpha=0.9)


plt.plot(lags_ssa[start_lag:maxlag], mean_correlation_ssa_constant_3_step_ki[start_lag:maxlag], 'o-', color='g', linewidth=2, label='SSA 3_step_ki', alpha=0.5)

plt.fill_between(lags_ssa[start_lag:maxlag],
                mean_correlation_ssa_constant_3_step_ki[start_lag:maxlag] - std_correlation_ssa_constant_3_step_ki[start_lag:maxlag],
                mean_correlation_ssa_constant_3_step_ki[start_lag:maxlag] + std_correlation_ssa_constant_3_step_ki[start_lag:maxlag],
                color='g', alpha=0.9)

# legend
plt.legend(loc='upper right')

In [ ]:
raise

In [ ]:
constant_elongation_rate = None
# constant_elongation_rate = 5

list_ribosome_trajectories, list_occupancy_output, ssa_complete,ssa_complete_second  = simulate_TASEP_SSA(ki, ke, gene_length, t_max,
                                time_interval_in_seconds=step_size_in_sec,
                                number_repetitions=2, 
                                first_probe_position_vector=first_probe_position_vector, 
                                second_probe_position_vector=second_probe_position_vector,
                                burnin_time=burnin_time,
                                constant_elongation_rate=constant_elongation_rate,
                                fast_output = False,) 

In [ ]:
selected_trajectory = 0

ribosome_trajectories = list_ribosome_trajectories[selected_trajectory]    
ribosome_trajectories = ribosome_trajectories[:,:]
intensity_vector_first_signal = ssa_complete[selected_trajectory,:]
if second_probe_position_vector is not None:
    intensity_vector_second_signal = ssa_complete_second[selected_trajectory,:]
else:
    intensity_vector_second_signal = None


In [ ]:
ki = 0.03  # Initiation rate
elongation_rate_constant = 5  # Elongation rates for positions 1 to N-1
number_repetitions = 50
burnin_time = 500
t_max = 360*5 #timePerturbationApplication + 25*60  # Maximum time
step_size_in_sec = 5 # 5

In [ ]:
dna_file_path.name.split('.')[0]
plasmid_name = dna_file_path.name.split('.')[0].replace('(','_').replace(')','')
plasmid_name

In [ ]:
str_ki = str(ki).replace('.','_')
str_k = str(elongation_rate_constant).replace('.','_')
fileNameGif = 'simulation_'+plasmid_name+'_ke_'+str_k+'_ki_'+str_ki 
plot_RibosomeMovement_and_Microscope(ribosome_trajectories, intensity_vector_first_signal, tag_positions_first_probe_vector, SecondIntensityVector=intensity_vector_second_signal, second_probePositions=tag_positions_second_probe_vector,FrameVelocity=10,fileNameGif=fileNameGif)

In [ ]:
raise

In [ ]:
import numpy as np

In [ ]:
ki_pre_min=0.02
ki_pre_max=0.08
ki_ini_min=0.02
ki_ini_max=0.08
ke_min=3.0
ke_max=7.0
dim_size = 30

In [ ]:
ke_values = np.round(np.linspace(ke_min, ke_max, dim_size),1)
print(ke_values)

In [ ]:
ki_pre_values = np.round( np.linspace(ki_pre_min, ki_pre_max, dim_size), 3)
# roun this linspace with 4 significant digits
print(ki_pre_values)


ki_ini_values = np.linspace(ki_ini_min, ki_ini_max, dim_size)
ke_values = np.linspace(ke_min, ke_max, dim_size)
# param_grid = [(float(ki_pre), float(ki_ini), float(ke))
#                 for ki_pre in ki_pre_values
#                 for ki_ini in ki_ini_values
#                 for ke in ke_values]
param_grid = [
(
    float("{:.3g}".format(ki_pre)),
    float("{:.3g}".format(ki_ini)),
    float("{:.3g}".format(ke))
)
for ki_pre in ki_pre_values
for ki_ini in ki_ini_values
for ke in ke_values]

In [ ]:
ki_pre_values

In [ ]:
param_grid